### BPE tokenizer Notes:
- The byte-level BPE is another approach. It simply converts the text into UTF-8 first, and treat it as a stream of bytes.
- This guarantees that any text encoded in UTF-8 can be encoded by the BPE. 
- This has been used in BERT-like models like RoBERTa, BART, and DeBERTa, and GPT-like models like GPT-2.[14][15][16]

- Even though block size is 8 ( or > 1) we have to train the network with no or just a single char input so that it can generate when we just have nothing to start with.


### Attention notes:
- Attention is a *communication mechanism*. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space, you have to provide that notion with a position dependent embedding.
- Each example across the batch dimension is processed independently, there is no communication across them, they never "talk" to each other.
- In an encoder attention block -- allow all tokens to communicate with each other. In a decoder block, the mask (tril) is needed so that a token only communicates with the past ones. Eg: Sentiment analysis. Future nodes can't talk to past, otherwise they give away the answer.
- "Self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", queries are from x, keys and values are from another source, separate source, nodes on the side with context. cross attention when there is a separate set of nodes where we'd like to pool information from. In principle, attention is very general.
- Normalize Qk(t) by root(dk) so that the attention values do not go into periphery of the softmax function, needed to keep the gradients from getting extremely small. dk = head_size. -- *Scaled Attention*.
- *Causal Attention*: Means the model can only look at the current and past tokens (not the future, can't cheat). This is helpful for generation. Non-causal attention is useful for understanding, classification etc.
- Self attention can't tolerate very high learning rates.

In [2]:
# Tiny shakespeare dataset: https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
# RNN effectiveness: https://karpathy.github.io/2015/05/21/rnn-effectiveness/

# tokenize just means convert the input space (vocabulary, etc) into a sequence of integers.

# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
# !wget  https://cs.stanford.edu/people/karpathy/char-rnn/linux.txt

In [56]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
import numpy as np
%matplotlib inline

In [57]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

#with open('linux.txt', 'r', encoding='utf-8') as f:
#    linux_code = f.read()

In [58]:


chars = sorted(list(set(text)))
''.join(chars)
vocab_size = len(chars)

In [59]:
# tokenizers: we can have a very long sequence of integers (as token output) with a very small vocabulary or
# a very large vocabulary with a very small sequence of integers as the encoder output.

In [60]:

# byte-pair encoding: https://en.wikipedia.org/wiki/Byte-pair_encoding

In [61]:
import tiktoken
encoder = tiktoken.get_encoding("gpt2")
encoder.encode("hi there")

[5303, 612]

In [62]:
# hyper params


batch_size = 32 # how many independent sequences will we process in parallel?

block_size = 8 # what is the maximum context length for predictions? # this is the context length !
max_iters = 5000
eval_interval = 500
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embed = 100


In [63]:
itos = {i:ch for i, ch in enumerate(chars)}
stoi = {ch:i for i, ch in enumerate(chars)}
# for i, s in enumerate(chars):
#     itos[i] = s
#     stoi[s] = i


encoder = lambda s: [stoi[x] for x in s]
decoder = lambda x: ''.join([itos[xx] for xx in x]) 

In [64]:
x1 = encoder("hi there")
print(x1)
print(decoder(x1))

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [65]:
# training and val split
data_len = len(text)
split_boundary = int(0.9 * data_len)
encoded_data = torch.tensor(encoder(text))

train_data = encoded_data[:split_boundary]
val_data = encoded_data[split_boundary:]

print(f"number of chars = {data_len}, split boundary = {split_boundary}, length of training data = {len(train_data)}, length of val = {len(val_data)}")


number of chars = 1115394, split boundary = 1003854, length of training data = 1003854, length of val = 111540


In [66]:
torch.manual_seed(1337)
#batch_size = 4

def get_sample_batch(split):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - block_size, (batch_size, ))
    xx = [data[i: i + block_size] for i in ix]
    yy = [data[i+1: i + block_size +1] for i in ix]
    return torch.stack(xx), torch.stack(yy)


xx, yy = get_sample_batch('train')

# for x, y in zip(xx, yy):
#     for j in range(len(x)):
#         print(f"when input is {x[:j+1]}    ----->    output is {y[j]}")

#print(xx)
#print(yy)

In [67]:
vocab_size, xx.size(), yy.size()
print(yy)

tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39],
        [43, 60, 43, 52,  1, 63, 43, 39],
        [43, 42,  8,  0, 25, 63,  1, 45],
        [42,  5, 57,  1, 57, 39, 49, 43],
        [57, 58, 63,  6,  1, 58, 46, 47],
        [ 1, 51, 39, 63,  1, 40, 43,  1],
        [46, 43,  1, 43, 39, 56, 57, 10],
        [58, 47, 53, 52, 12,  1, 37, 53],
        [56, 43,  1, 21,  1, 41, 39, 51],
        [39, 52, 63,  1, 47, 58, 57, 43],
        [53, 63,  1, 42, 47, 42,  1, 57],
        [51,  1, 39, 44, 56, 39, 47, 42],
        [24, 21, 38, 13, 14, 17, 32, 20],
        [39, 52, 42,  1, 45, 43, 50, 42],
        [58, 46, 39, 58,  1, 42, 53,  1],
        [61, 53, 59, 50, 42,  1, 21,  1],
        [57, 40, 39, 52, 42,  1, 40, 47],
        [42,  8,  0,  0, 23, 21, 26, 19],
        [53, 42, 57,  0, 23, 43, 43, 54],
        [ 1, 61, 39, 57,  1, 51, 53, 56],
        [49, 12,  1, 27,  1, 58, 5

In [68]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_sample_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [69]:
class Head(nn.Module):
    """ Implements a single head of self-attention"""

    # X = (batch_size, seq_len, embed_dim)
    def __init__(self, head_size, embed_dim, is_causal = True):
        super().__init__()
        self.head_size = head_size
        self.embed_dim = embed_dim

        self.query = nn.Linear(embed_dim, head_size, bias=False)
        self.key = nn.Linear(embed_dim, head_size, bias=False)
        self.value = nn.Linear(embed_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.is_causal = is_causal

    def forward(self, x):
        B, T, C = x.shape
        Q = self.query(x)  # B, T, hs
        K = self.key(x)  # B, T, hs
        V = self.value(x) # B, T, hs

        # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = Q @ K.transpose(-2, -1) * (head_size ** -0.5)
        if self.is_causal:   # mask out the top triangle. 
            wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        wei = F.softmax(wei, dim=-1)# (B, T, T)
        out = wei @ V # (B, T, T) @ (B, T, hs)
        return out



#lass MultiheadAttention(nn.Module)
         
         

In [85]:
import torch.nn as nn

# C = number of channels == embedding dim or for hidden layers its the larger last dimension
# T = sequence length or block_size or context_length
# B = batch size or length

class BigramLM(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding = nn.Embedding(block_size, n_embed)
        self.sa_head = Head(n_embed, n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # B, T, C = n_embed
        pos_emb = self.position_embedding(torch.arange(T, device=device)) # B, T, C

        x = tok_emb + pos_emb

        x = self.sa_head(x)
        #print(x.shape, tok_emb.shape, pos_emb.shape, idx.shape)
        logits = self.lm_head(x)
        B, T, C = logits.shape

        lg_view = logits
        if targets == None:
            loss = None
        else:
            t = targets.view(B*T)
            lg_view = logits.view(B*T, C)
            loss = F.cross_entropy(lg_view, t)
            
        return lg_view, loss
        
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices of current context
        print(f"idx.shape = {idx.shape}")
        for _ in range(max_new_tokens):
            idx_cond = idx[: -block_size:]
            logits, loss = self(idx_cond)
            print(logits, loss)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            idx_next = torch.multinomial(probs, num_samples=1)
            print(f"idx_next = {idx_next.shape}, {idx_next, probs}")
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLM(vocab_size)
m = model.to(device)
logits, loss = m(xx, yy)


#print(logits, loss)

print(decoder(m.generate(torch.tensor([[0, 0, 0, 0, 0, 0, 0]]), 100)[0].tolist()))

idx.shape = torch.Size([1, 7])
tensor([], size=(0, 7, 65), grad_fn=<ViewBackward0>) None
idx_next = torch.Size([0, 1]), (tensor([], size=(0, 1), dtype=torch.int64), tensor([], size=(0, 65), grad_fn=<SoftmaxBackward0>))


RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 1 but got size 0 for tensor number 1 in the list.

In [86]:
# create a pytorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [87]:
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_sample_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)


step 0: train loss 4.2235, val loss 4.2303
step 500: train loss 2.5187, val loss 2.5431
step 1000: train loss 2.4801, val loss 2.4974
step 1500: train loss 2.4309, val loss 2.4692
step 2000: train loss 2.4068, val loss 2.4324
step 2500: train loss 2.3873, val loss 2.4263
step 3000: train loss 2.3919, val loss 2.4125
step 3500: train loss 2.3745, val loss 2.4118
step 4000: train loss 2.3775, val loss 2.3906
step 4500: train loss 2.3681, val loss 2.3912


### Ledger of architecture and loss values
- Single head, head_size =100, linear layer = 100, block_size = 8
  - 5000 steps: train loss 2.3681, val loss 2.3912
  - attention layers need a lower learning rate 1e-3 

In [25]:
print(decoder(m.generate(context, max_new_tokens=500)[0].tolist()))


Levld;.
OqDChe s sHwdishavnof KpGBEGLOusU.OK:
FounKzU
AndulivoowCgm bupof'qmod: wy rt aeELTI?W,
BYO:Nhm '
YCI as s atoranUK?OUK: steis,
SForngbowehilxz
YOKIZ3Zz't gurtbam wWky& our ne
W?
Pkyesed h y-t?WiRKVItpun mESind,&bes;
AGhnyfzM3ZQGLION?murand .3Swin,lofusxiXLferer, lay;----!IONKIDgidmToQbn wrez$nouplwilq&e n.
By aen
Opleseat; hsthe,IAwa,VIsave isple ?OYCod likIXSBY: ckendavFoumSwxPr-JAUMAN'e d dCASgell
Tavera-!Th ledoFodit ck
O,zBATh3SAYSMX?MIOKIjugghiU&Vphesh her3bZlLorWherthN Nclxe&Y3rXr


NameError: name 'attn_weights' is not defined

In [28]:
# Scaled attention illustration.
B, T, C = 4, 8, 32
head_size = 16
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)

In [29]:
k.mean(), k.var(), q.mean(), q.var()

(tensor(0.0003), tensor(0.8805), tensor(0.0464), tensor(0.9541))

In [30]:
wei = q@k.transpose(1,2)
wei.mean(), wei.var(), head_size



(tensor(-0.1432), tensor(14.5549), 16)

In [31]:
wei2 = q @k.transpose(1,2) / (head_size ** 0.5) 

wei2.mean(), wei2.var()

(tensor(-0.0358), tensor(0.9097))

In [ ]:
# Build single head attention layer and add to bigram.py 

In [92]:
z1 = torch.randn((4, 5))
z2 = torch.randn((4, 5))
z1, z2

(tensor([[-0.4909, -0.7850, -0.5818, -0.5959,  0.2850],
         [ 1.9446,  0.4221, -0.2017, -0.0036,  0.9279],
         [-1.2379,  0.1841, -0.7239, -1.2522,  0.4589],
         [-1.8938, -0.7853,  0.2466,  0.0422, -0.7050]]),
 tensor([[ 0.4491, -0.4149,  0.9981, -0.3319,  0.0734],
         [-0.7052, -1.1703,  1.8885,  0.8651,  0.3030],
         [ 1.3659, -1.0755, -1.3451, -0.1839,  0.2542],
         [ 1.0291,  0.5244,  0.5959, -1.0257,  1.3285]]))

In [98]:
torch.stack((z1, z2), dim=2)

tensor([[[-0.4909,  0.4491],
         [-0.7850, -0.4149],
         [-0.5818,  0.9981],
         [-0.5959, -0.3319],
         [ 0.2850,  0.0734]],

        [[ 1.9446, -0.7052],
         [ 0.4221, -1.1703],
         [-0.2017,  1.8885],
         [-0.0036,  0.8651],
         [ 0.9279,  0.3030]],

        [[-1.2379,  1.3659],
         [ 0.1841, -1.0755],
         [-0.7239, -1.3451],
         [-1.2522, -0.1839],
         [ 0.4589,  0.2542]],

        [[-1.8938,  1.0291],
         [-0.7853,  0.5244],
         [ 0.2466,  0.5959],
         [ 0.0422, -1.0257],
         [-0.7050,  1.3285]]])

In [96]:
t = [ [[-0.4909, -0.7850, -0.5818, -0.5959,  0.2850], [ 0.4491, -0.4149,  0.9981, -0.3319,  0.0734]], 
      [[ 1.9446,  0.4221, -0.2017, -0.0036,  0.9279], [-0.7052, -1.1703,  1.8885,  0.8651,  0.3030]]]
